# Wikipedia airborne-radar knowledge graph with Neo4j

This notebook uses the repository's `combat_id_calibration.graph_ingest` module to build a local Neo4j knowledge graph from Wikipedia pages for the MiG-29, the Ukrainian Air Force, the Russian Air Force, the Bars radar, and representative Russian or Israeli airborne radars.

The existing ingestion module performs the core workflow: fetch Wikipedia HTML, extract readable text, chunk documents, ask a local Ollama model to return auditable JSON facts, optionally write facts to JSONL, and populate Neo4j with `Entity`, `Source`, `FACT`, and `MENTIONED_IN` records.

> Wikipedia is a convenient public source, but it is not authoritative. Review `facts-jsonl` output before using extracted relationships for combat-identification scoring or calibration.


## 1. Install and runtime prerequisites

From the repository root, install the graph extra and make sure Ollama is running with the configured model:

```bash
python -m pip install -e .[graph]
ollama pull qwen3.5:9b
ollama serve
```

If your notebook starts in `notebooks/`, the setup cell below adds the repository root to `sys.path` so it imports the local module under development.


In [ ]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from combat_id_calibration.graph_ingest import (
    DEFAULT_MODEL,
    DEFAULT_OLLAMA_URL,
    extract_facts,
    load_documents,
    populate_neo4j,
    write_facts_jsonl,
)


## 2. Configure Wikipedia sources

The first four URLs satisfy the explicitly requested pages. The remaining URLs add representative Russian and Israeli airborne radar pages so the resulting graph has more radar-specific evidence.


In [ ]:
NEO4J_URI = 'bolt://localhost:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = None  # replace with your Neo4j password before running the population cell
NEO4J_DATABASE = None
FACTS_JSONL = REPO_ROOT / 'wikipedia_airborne_radars_facts.jsonl'
MODEL = DEFAULT_MODEL
OLLAMA_URL = DEFAULT_OLLAMA_URL
MAX_CHARS = 6000
OVERLAP = 500

WIKIPEDIA_URLS = [
    'https://en.wikipedia.org/wiki/Mikoyan_MiG-29',
    'https://en.wikipedia.org/wiki/Ukrainian_Air_Force',
    'https://en.wikipedia.org/wiki/Russian_Air_Force',
    'https://en.wikipedia.org/wiki/Bars_radar',

    # Additional Russian airborne radars
    'https://en.wikipedia.org/wiki/Irbis-E',
    'https://en.wikipedia.org/wiki/Zhuk_(radar)',
    'https://en.wikipedia.org/wiki/Mech_radar',
    # Additional Israeli airborne radars
    'https://en.wikipedia.org/wiki/EL/M-2032',
    'https://en.wikipedia.org/wiki/EL/W-2085',
    'https://en.wikipedia.org/wiki/EL/M-2052',
]


## 3. Load Wikipedia documents with `graph_ingest.py`

`load_documents` delegates each Wikipedia URL to `read_wikipedia`, preserving a deterministic source ID, source type, locator URL, article title, and extracted article text.


In [ ]:
documents = load_documents(pdf_paths=[], wikipedia_urls=WIKIPEDIA_URLS)
print(f'Loaded {len(documents)} Wikipedia documents')
for document in documents:
    print(f'- {document.title}: {len(document.text):,} characters from {document.locator}')


## 4. Extract auditable facts with Ollama

This cell calls the repository ingestion module's `extract_facts`, which chunks each source and applies the module's constrained JSON extraction prompt. Keep `FACTS_JSONL` under review; it is the audit artifact to inspect before trusting the Neo4j graph.


In [ ]:
facts = extract_facts(
    documents,
    model=MODEL,
    ollama_url=OLLAMA_URL,
    max_chars=MAX_CHARS,
    overlap=OVERLAP,
)
write_facts_jsonl(facts, FACTS_JSONL)
print(f'Extracted {len(facts)} facts')
print(f'Wrote review file: {FACTS_JSONL}')


## 5. Populate Neo4j with `graph_ingest.py`

`populate_neo4j` creates the same schema used by the repository CLI:

- `Entity(id, name)`
- `Source(id, source_type, locator)`
- `FACT(predicate, source_id, evidence, confidence)`
- `MENTIONED_IN`

If you want a clean graph, clear the target Neo4j database before running this population cell.

If `NEO4J_PASSWORD` is left as `None` or an empty string, the population cell stops before opening a Neo4j driver and asks you to set the password.


In [ ]:
populate_neo4j(facts, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)
print(f'Populated Neo4j database: {NEO4J_URI}')


## 6. Query examples

The queries below inspect the graph generated by `populate_neo4j`. They intentionally work with the generic `Entity`/`FACT` schema from `graph_ingest.py` instead of introducing a separate notebook-only schema.


In [ ]:
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
session_kwargs = {'database': NEO4J_DATABASE} if NEO4J_DATABASE else {}
session = driver.session(**session_kwargs)

def show(query: str, params: dict | None = None):
    result = session.run(query, params or {})
    rows = pd.DataFrame([record.data() for record in result])
    display(rows)
    return rows


In [ ]:
show('''
MATCH (source:Source)
RETURN source.source_type AS source_type, source.locator AS locator
ORDER BY locator
''')


In [ ]:
show('''
MATCH (subject:Entity)-[fact:FACT]->(object:Entity)
WHERE lower(subject.name) CONTAINS 'radar'
   OR lower(object.name) CONTAINS 'radar'
   OR lower(fact.predicate) CONTAINS 'sensor'
RETURN subject.name AS subject, fact.predicate AS predicate, object.name AS object,
       fact.confidence AS confidence, fact.evidence AS evidence
ORDER BY confidence DESC
LIMIT 50
''')


In [ ]:
show('''
MATCH (subject:Entity)-[fact:FACT]->(object:Entity)
WHERE lower(subject.name) CONTAINS 'mig-29'
   OR lower(object.name) CONTAINS 'mig-29'
   OR lower(subject.name) CONTAINS 'air force'
   OR lower(object.name) CONTAINS 'air force'
RETURN subject.name AS subject, fact.predicate AS predicate, object.name AS object,
       fact.confidence AS confidence, fact.evidence AS evidence
ORDER BY confidence DESC
LIMIT 50
''')


## 7. Equivalent CLI command

The same module is also exposed by the repository CLI. This notebook form is useful for iterative review, while the command below is better for repeatable batch runs.


In [ ]:
cli = [
    'python -m combat_id_calibration ingest-graph',
    f'  --neo4j-uri {NEO4J_URI}',
    f'  --neo4j-user {NEO4J_USER}',
    '  --neo4j-password "$NEO4J_PASSWORD"',
    f'  --facts-jsonl {FACTS_JSONL}',
    f'  --model {MODEL}',
    f'  --ollama-url {OLLAMA_URL}',
    *[f'  --wikipedia {url}' for url in WIKIPEDIA_URLS],
]
print(' \
'.join(cli))
